# Crypto Exchange API Tester

Manual notebook for testing the project Bybit and OKX API integration paths.

Run this notebook from the `backend` directory with the backend virtual environment kernel. It uses the existing project clients and normalizers:

- `core.crypto_exchange_clients.BybitClient`
- `core.crypto_exchange_clients.OKXClient`
- `core.crypto_exchange_import.normalize_bybit_spot_execution`
- `core.crypto_exchange_import.normalize_okx_spot_fill`
- optional `core.broker_api_utils.BybitAPI` / `OKXAPI` using credentials stored in the app database

The notebook does not print API secrets and does not write transactions to the database unless `ALLOW_DB_WRITES = True` is set explicitly.

## Credential Setup

Create read-only credentials in the exchange UI. Do not enable trading or withdrawal permissions.

Bybit:

- Create either a mainnet key or a testnet key. Testnet keys are separate from mainnet keys.
- Required values: API key and API secret.
- Required read permissions: account/wallet read, transaction log read, order/execution history read for the categories you want to test.
- Disable withdrawals and trading permissions.
- Restrict by IP if your network setup allows it.
- Environment variables used by this notebook:
  - `BYBIT_API_KEY`
  - `BYBIT_API_SECRET`
  - `BYBIT_TESTNET=1` for testnet, otherwise `0`
  - optional `BYBIT_ACCOUNT_TYPE=UNIFIED`
  - optional `BYBIT_CATEGORY=spot`

OKX:

- Create a read-only API key. OKX also requires the passphrase created with the key.
- Required values: API key, API secret, passphrase.
- Required read permissions: account read and trade/fills history read. For future rewards/transfers testing, also allow read access to funding/account bills.
- Disable withdrawals and trading permissions.
- Restrict by IP if your network setup allows it.
- For demo/simulated testing, create or use demo trading credentials and set `OKX_SIMULATED_TRADING=1`.
- Environment variables used by this notebook:
  - `OKX_API_KEY`
  - `OKX_API_SECRET`
  - `OKX_PASSPHRASE`
  - `OKX_SIMULATED_TRADING=1` for demo/simulated trading, otherwise `0`

Optional database-token path:

- Create a `Brokers` row for Bybit or OKX in the app.
- Store the token through User Settings -> Broker API credentials.
- Create a broker account under that broker.
- Set `PM_USER_ID` and `PM_ACCOUNT_ID` below if you want to test the project `BrokerAPI` adapter path.

In [ ]:
import os
import sys
from datetime import datetime, timedelta, timezone
from pprint import pprint

# Keep the backend directory importable when the notebook is opened from here.
backend_dir = os.path.abspath('.')
if backend_dir not in sys.path:
    sys.path.append(backend_dir)

from notebook_setup import setup_django

setup_django()

from common.models import Accounts, Transactions
from core.broker_api_utils import BybitAPI, OKXAPI
from core.crypto_exchange_clients import BybitClient, CryptoExchangeAPIError, OKXClient
from core.crypto_exchange_import import (
    normalize_bybit_spot_execution,
    normalize_okx_spot_fill,
    persist_crypto_exchange_event,
)
from users.models import CustomUser

In [ ]:
def env_bool(name, default=False):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y'}


def masked(value):
    if not value:
        return '<missing>'
    if len(value) <= 8:
        return '<set>'
    return f'{value[:4]}...{value[-4:]}'


def require_env(*names):
    missing = [name for name in names if not os.getenv(name)]
    if missing:
        raise RuntimeError(
            'Missing environment variables: ' + ', '.join(missing)
        )


def date_range_ms(days=7):
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days)
    return str(int(start.timestamp() * 1000)), str(int(end.timestamp() * 1000))


LOOKBACK_DAYS = int(os.getenv('CRYPTO_API_LOOKBACK_DAYS', '7'))
ALLOW_DB_WRITES = env_bool('CRYPTO_API_TEST_ALLOW_DB_WRITES', False)

print('Credential preview:')
print('  BYBIT_API_KEY:', masked(os.getenv('BYBIT_API_KEY')))
print('  BYBIT_TESTNET:', env_bool('BYBIT_TESTNET', False))
print('  OKX_API_KEY:', masked(os.getenv('OKX_API_KEY')))
print('  OKX_SIMULATED_TRADING:', env_bool('OKX_SIMULATED_TRADING', False))
print('  LOOKBACK_DAYS:', LOOKBACK_DAYS)
print('  ALLOW_DB_WRITES:', ALLOW_DB_WRITES)

## Bybit Direct Client Smoke Tests

These cells test signed private requests directly against Bybit using environment credentials. The first cell verifies account access; later cells fetch recent spot executions and transaction log rows.

In [ ]:
require_env('BYBIT_API_KEY', 'BYBIT_API_SECRET')

bybit = BybitClient(
    api_key=os.environ['BYBIT_API_KEY'],
    api_secret=os.environ['BYBIT_API_SECRET'],
    testnet=env_bool('BYBIT_TESTNET', False),
)

bybit_account_type = os.getenv('BYBIT_ACCOUNT_TYPE', 'UNIFIED')
wallet = bybit.get_private(
    '/v5/account/wallet-balance',
    {'accountType': bybit_account_type},
)

print('Bybit wallet response keys:', wallet.keys())
pprint(wallet.get('result', {}))

In [ ]:
start_ms, end_ms = date_range_ms(LOOKBACK_DAYS)
bybit_category = os.getenv('BYBIT_CATEGORY', 'spot')
bybit_execution_params = {
    'category': bybit_category,
    'startTime': start_ms,
    'endTime': end_ms,
}

bybit_executions = []
try:
    bybit_executions = list(bybit.iter_executions(bybit_execution_params))
except CryptoExchangeAPIError as exc:
    print('Bybit execution fetch failed:', exc)

print(f'Fetched {len(bybit_executions)} Bybit execution rows')
if bybit_executions:
    pprint(bybit_executions[0])

In [ ]:
bybit_log_params = {
    'accountType': bybit_account_type,
    'startTime': start_ms,
    'endTime': end_ms,
}

bybit_log_rows = []
try:
    bybit_log_rows = list(bybit.iter_transaction_log(bybit_log_params))
except CryptoExchangeAPIError as exc:
    print('Bybit transaction-log fetch failed:', exc)

print(f'Fetched {len(bybit_log_rows)} Bybit transaction log rows')
if bybit_log_rows:
    pprint(bybit_log_rows[0])

In [ ]:
normalized_bybit_events = []
for payload in bybit_executions[:5]:
    try:
        normalized_bybit_events.append(normalize_bybit_spot_execution(payload))
    except Exception as exc:
        print('Could not normalize Bybit execution:')
        pprint(payload)
        print(exc)

print(f'Normalized {len(normalized_bybit_events)} Bybit events')
if normalized_bybit_events:
    pprint(normalized_bybit_events[0])

## OKX Direct Client Smoke Tests

These cells test signed private requests directly against OKX using environment credentials. The first cell verifies account access; later cells fetch recent SPOT fills and normalize them.

In [ ]:
require_env('OKX_API_KEY', 'OKX_API_SECRET', 'OKX_PASSPHRASE')

okx = OKXClient(
    api_key=os.environ['OKX_API_KEY'],
    api_secret=os.environ['OKX_API_SECRET'],
    passphrase=os.environ['OKX_PASSPHRASE'],
    simulated_trading=env_bool('OKX_SIMULATED_TRADING', False),
)

balance = okx.get_private('/api/v5/account/balance')
print('OKX balance response keys:', balance.keys())
pprint(balance.get('data', [])[:1])

In [ ]:
start_ms, end_ms = date_range_ms(LOOKBACK_DAYS)
okx_fill_params = {
    'instType': 'SPOT',
    'begin': start_ms,
    'end': end_ms,
}

okx_fills = []
try:
    okx_fills = list(okx.iter_fills_history(okx_fill_params))
except CryptoExchangeAPIError as exc:
    print('OKX fills fetch failed:', exc)

print(f'Fetched {len(okx_fills)} OKX fill rows')
if okx_fills:
    pprint(okx_fills[0])

In [ ]:
normalized_okx_events = []
for payload in okx_fills[:5]:
    try:
        normalized_okx_events.append(normalize_okx_spot_fill(payload))
    except Exception as exc:
        print('Could not normalize OKX fill:')
        pprint(payload)
        print(exc)

print(f'Normalized {len(normalized_okx_events)} OKX events')
if normalized_okx_events:
    pprint(normalized_okx_events[0])

## Optional: Test The Project BrokerAPI Adapter Path

Use this when credentials have been stored through User Settings and a broker account exists in the database. This exercises the same adapter path used by Direct Import.

Set:

- `PM_USER_ID`
- `PM_ACCOUNT_ID`
- optional `CRYPTO_API_LOOKBACK_DAYS`

This cell fetches normalized events only. It does not persist transactions.

In [ ]:
require_env('PM_USER_ID', 'PM_ACCOUNT_ID')

user = CustomUser.objects.get(id=int(os.environ['PM_USER_ID']))
account = Accounts.objects.select_related('broker').get(id=int(os.environ['PM_ACCOUNT_ID']))

has_bybit = account.broker.bybit_tokens.filter(user=user, is_active=True).exists()
has_okx = account.broker.okx_tokens.filter(user=user, is_active=True).exists()
if has_bybit:
    adapter = BybitAPI()
elif has_okx:
    adapter = OKXAPI()
else:
    raise RuntimeError('Selected broker has no active Bybit or OKX token for this user')

date_to = datetime.now(timezone.utc).date().isoformat()
date_from = (datetime.now(timezone.utc).date() - timedelta(days=LOOKBACK_DAYS)).isoformat()

await adapter.connect(user)
adapter_events = []
async for event in adapter.get_transactions(account, date_from=date_from, date_to=date_to):
    adapter_events.append(event)
await adapter.disconnect()

print(f'Fetched {len(adapter_events)} normalized events through {adapter.__class__.__name__}')
if adapter_events:
    pprint(adapter_events[0])

## Optional: Persist One Normalized Event

This writes `Transactions` rows. Keep `CRYPTO_API_TEST_ALLOW_DB_WRITES=0` unless you are intentionally testing persistence in a disposable database or with a transaction you are prepared to delete.

The importer is idempotent by provider/account/event id, but this is still a database write.

In [ ]:
if not ALLOW_DB_WRITES:
    raise RuntimeError(
        'Database writes are disabled. Set CRYPTO_API_TEST_ALLOW_DB_WRITES=1 to run this cell.'
    )

require_env('PM_USER_ID', 'PM_ACCOUNT_ID')
user = CustomUser.objects.get(id=int(os.environ['PM_USER_ID']))
account = Accounts.objects.get(id=int(os.environ['PM_ACCOUNT_ID']))
events_to_persist = adapter_events or normalized_bybit_events or normalized_okx_events
if not events_to_persist:
    raise RuntimeError('No normalized events available to persist')

created = persist_crypto_exchange_event(events_to_persist[0], user, account)
print(f'Created {len(created)} Transactions rows')
for tx in created:
    print(tx.id, tx.type, tx.security, tx.quantity, tx.price, tx.import_provider, tx.import_event_id)

## Troubleshooting Notes

- `Missing environment variables`: set the variables in the shell before starting Jupyter, then restart the kernel.
- Bybit `permission denied` or similar: check that the key has read access for account, wallet, transaction log, and execution history.
- Bybit testnet failures with mainnet keys: testnet and mainnet keys are not interchangeable.
- OKX `Invalid Sign`: verify the API secret and passphrase exactly. Passphrases are user-defined and case-sensitive.
- OKX simulated failures: set `OKX_SIMULATED_TRADING=1` only for demo/simulated credentials.
- Empty results are valid if there were no fills/log entries in the selected date range. Increase `CRYPTO_API_LOOKBACK_DAYS`.
- Keep raw payloads when normalization fails; those payloads are the best fixtures for extending the normalizers.